In [1]:
from pathlib import Path
from typing import Any
import sys

sys.path.insert(0, "/Users/chester/Documents/cc-net")

import graphlearning as gl
import numpy as np
import scipy.sparse as sp
import torch
import yaml
from sklearn.cluster import KMeans
from torch_geometric.data import Data

import models.models as Models
from models.model_utils import project_l2

/Users/chester/Documents/cc-net/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _to_numpy_labels(labels: torch.Tensor | np.ndarray) -> np.ndarray:
    if isinstance(labels, torch.Tensor):
        return labels.detach().cpu().numpy().astype(np.int64)
    return np.asarray(labels, dtype=np.int64)


def kmeans_readout(
    embeddings: torch.Tensor | np.ndarray,
    num_clusters: int,
    seed: int,
    n_init: int = 10,
) -> np.ndarray:
    X = embeddings.detach().cpu().numpy() if isinstance(embeddings, torch.Tensor) else np.asarray(embeddings)
    km = KMeans(n_clusters=num_clusters, random_state=seed, n_init=n_init)
    return km.fit_predict(X).astype(np.int64)


def clustering_accuracy(pred_labels: np.ndarray, true_labels: torch.Tensor | np.ndarray) -> float:
    true_np = _to_numpy_labels(true_labels)
    return float(gl.clustering.clustering_accuracy(pred_labels.astype(np.int64), true_np))


def embedding_clustering_accuracy(
    embeddings: torch.Tensor | np.ndarray,
    true_labels: torch.Tensor | np.ndarray,
    num_clusters: int,
    seed: int,
    n_init: int = 10,
) -> tuple[float, np.ndarray]:
    pred = kmeans_readout(embeddings=embeddings, num_clusters=num_clusters, seed=seed, n_init=n_init)
    acc = clustering_accuracy(pred_labels=pred, true_labels=true_labels)
    return acc, pred


def filter_subset_classes(
    X: np.ndarray,
    y: np.ndarray,
    classes: np.ndarray,
    seed: int,
    max_points_per_class: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    selected_indices: list[np.ndarray] = []

    for cls in classes:
        cls_idx = np.flatnonzero(y == cls)
        if cls_idx.size == 0:
            raise ValueError(f"No samples found for class {int(cls)}.")
        if max_points_per_class is not None and cls_idx.size > max_points_per_class:
            cls_idx = rng.choice(cls_idx, size=max_points_per_class, replace=False)
        selected_indices.append(np.asarray(cls_idx, dtype=np.int64))

    all_idx = np.concatenate(selected_indices, axis=0)
    rng.shuffle(all_idx)
    return X[all_idx], y[all_idx]


def build_knn_weight_matrix(X: np.ndarray, k_neighbors: int, kernel: str) -> sp.csr_matrix:
    W = gl.weightmatrix.knn(X, k=k_neighbors, kernel=kernel).tocsr()
    W.setdiag(0)
    W.eliminate_zeros()
    return W


def knn_to_pyg_data(X: np.ndarray, y: np.ndarray, W: sp.csr_matrix) -> Data:
    C = sp.triu(W, k=1).tocoo()
    edge_index = torch.from_numpy(np.vstack([C.row, C.col]).astype(np.int64))
    edge_attr = torch.from_numpy(C.data.astype(np.float32))
    return Data(
        x=torch.from_numpy(X.astype(np.float32)),
        y=torch.from_numpy(y.astype(np.int64)),
        edge_index=edge_index,
        edge_attr=edge_attr,
    )


def build_full_graph(cfg: dict[str, Any]) -> Data:
    dataset_name = cfg["dataset"]["name"]
    metric = cfg["dataset"]["metric"]
    seed = int(cfg["split"]["seed"])
    k_neighbors = int(cfg["graph"]["k_neighbors"])
    kernel = str(cfg["graph"]["kernel"])

    X, y = gl.datasets.load(dataset_name, metric)
    X = np.asarray(X)
    y = np.asarray(y).astype(np.int64)

    full_max_points_per_class = cfg.get("evaluation", {}).get("max_points_per_class")
    if full_max_points_per_class is not None:
        classes = np.unique(y)
        X, y = filter_subset_classes(
            X=X,
            y=y,
            classes=classes,
            seed=seed,
            max_points_per_class=int(full_max_points_per_class),
        )

    W = build_knn_weight_matrix(X, k_neighbors=k_neighbors, kernel=kernel)
    return knn_to_pyg_data(X, y, W)


def _looks_like_state_dict(candidate: Any) -> bool:
    if not isinstance(candidate, dict) or not candidate:
        return False
    return all(isinstance(k, str) and torch.is_tensor(v) for k, v in candidate.items())

def normalize_model_output(model_output: Any) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
    if not isinstance(model_output, tuple):
        raise TypeError(f"Expected model output tuple, got {type(model_output)}")
    if len(model_output) == 2:
        U, P = model_output
        return U, P, None
    if len(model_output) >= 3:
        U, P, U_latent = model_output[:3]
        return U, P, U_latent
    raise ValueError("Model returned an empty tuple.")

def load_model(checkpoint_path: Path, cfg: dict[str, Any], device: torch.device) -> torch.nn.Module:
    ckpt = torch.load(checkpoint_path, map_location=device)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model_name = ckpt.get("model_name", cfg["model"]["name"])
        model_cfg = ckpt.get("model_cfg", cfg["model"]["cfg"])
        state = ckpt["model_state_dict"]
    elif _looks_like_state_dict(ckpt):
        model_name = cfg["model"]["name"]
        model_cfg = cfg["model"]["cfg"]
        state = ckpt
    else:
        raise ValueError(
            "Unsupported checkpoint format. Provide a .pt with model_state_dict, or a plain state_dict."
        )

    model_class = getattr(Models, model_name)
    model = model_class(**model_cfg).float().to(device)
    model.load_state_dict(state)
    model.eval()
    return model


def forward_graph(model: torch.nn.Module, graph: Data) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
    src = graph.edge_index[0]
    dst = graph.edge_index[1]
    x = graph.x.float()
    e_init = (x[src] - x[dst]).float()
    out = model(h=x, e=e_init, edge_index=graph.edge_index, w=graph.edge_attr.float(), x=x)
    return normalize_model_output(out)


def graph_divergence(msg: torch.Tensor, src: torch.Tensor, dst: torch.Tensor, num_nodes: int) -> torch.Tensor:
    out = torch.zeros(num_nodes, msg.size(-1), dtype=msg.dtype, device=msg.device)
    out.index_add_(0, src, msg)
    out.index_add_(0, dst, -msg)
    return out


def vanilla_pdhg(
    X: torch.Tensor,
    edge_index: torch.Tensor,
    w: torch.Tensor,
    lam: float = 1.0,
    iters: int = 200,
    tau: float = 0.35,
    sigma: float = 0.35,
    use_extrapolation: bool = True,
    theta: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    src, dst = edge_index
    sqrtw = w.sqrt()
    n, d = X.shape

    U = X.clone()
    U_bar = U.clone()
    P = torch.zeros(src.numel(), d, dtype=X.dtype, device=X.device)

    theta_eff = float(theta if use_extrapolation else 0.0)
    r = lam * sqrtw

    for _ in range(iters):
        diff_bar = U_bar[src] - U_bar[dst]
        P_new = project_l2(P + tau * (sqrtw[:, None] * diff_bar), r)

        div = graph_divergence(sqrtw[:, None] * P_new, src, dst, n)
        U_new = (U + sigma * (X - div)) / (1.0 + sigma)

        U_prev = U
        U = U_new
        P = P_new
        U_bar = U + theta_eff * (U - U_prev)

    return U, P


In [4]:
CONFIG_PATH = Path("./experiments/configs/mnist_subset_full_eval.yaml").resolve()
CHECKPOINT_PATH = Path("./outputs/experiments/mnist_train_03/lambda_1/last.pt").resolve()  # change if needed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PDHG_ITERS = 600
PDHG_LAMBDA = None  # set float explicitly, or keep None to use model lambda from config
USE_EXTRAPOLATION = True
THETA = 1.0

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

seed = int(cfg["train"]["seed"])
kmeans_n_init = int(cfg["evaluation"].get("kmeans_n_init", 10))
full_num_clusters = int(cfg["evaluation"]["full_num_clusters"])
tau = float(cfg["model"]["cfg"]["processor_cfg"]["cfg"].get("tau", 0.35))
sigma = float(cfg["model"]["cfg"]["processor_cfg"]["cfg"].get("sigma", 0.35))
if PDHG_LAMBDA is None:
    PDHG_LAMBDA = float(cfg["model"]["cfg"].get("lam", 1.0))

np.random.seed(seed)
torch.manual_seed(seed)

full_graph = build_full_graph(cfg).to(DEVICE)
model = load_model(CHECKPOINT_PATH, cfg=cfg, device=DEVICE)

AttributeError: 'str' object has no attribute 'resolve'

In [ ]:
with torch.no_grad():
    x = full_graph.x.float()

    # (1) KMeans on raw input embeddings
    acc_kmeans, _ = embedding_clustering_accuracy(
        embeddings=x,
        true_labels=full_graph.y,
        num_clusters=full_num_clusters,
        seed=seed,
        n_init=kmeans_n_init,
    )

    # (2) Vanilla PDHG on raw input embeddings
    U_pdhg, _ = vanilla_pdhg(
        X=x,
        edge_index=full_graph.edge_index,
        w=full_graph.edge_attr.float(),
        lam=PDHG_LAMBDA,
        iters=PDHG_ITERS,
        tau=tau,
        sigma=sigma,
        use_extrapolation=USE_EXTRAPOLATION,
        theta=THETA,
    )
    acc_pdhg, _ = embedding_clustering_accuracy(
        embeddings=U_pdhg,
        true_labels=full_graph.y,
        num_clusters=full_num_clusters,
        seed=seed,
        n_init=kmeans_n_init,
    )

    # (3) Neural model output
    U_model, _ = forward_graph(model, full_graph)
    acc_model, _ = embedding_clustering_accuracy(
        embeddings=U_model,
        true_labels=full_graph.y,
        num_clusters=full_num_clusters,
        seed=seed,
        n_init=kmeans_n_init,
    )

results = [
    {"method": "kmeans_input", "accuracy": float(acc_kmeans)},
    {"method": "pdhg_input", "accuracy": float(acc_pdhg)},
    {"method": "nn_output", "accuracy": float(acc_model)},
]

for row in results:
    print(f"{row['method']}: {row['accuracy']:.6f}")
